# PCAP Statistics

Extração completa do diretório **01-12**, independente dos CSVs do CICFlowMeter.
Cada linha representa uma janela global de todo o tráfego IPv4 observado, sem agrupamento por IP.
Não há rotulagem, seleção de ataques, balanceamento, imputação ou criação de cenários nesta etapa.
A referência desta implementação é o projeto `Atual-2.zip`.


## 1. Dependencies and Configuration

Execute no computador com acesso ao disco E:. Se necessário, descomente a instalação e execute-a uma vez.
`maxPackets=None` processa todos os pacotes de todos os arquivos; um inteiro limita globalmente os pacotes processados, incluindo não IPv4.
A intercalação mantém um pacote de antecipação por arquivo, fora da contagem de observações processadas.


In [ ]:
# %pip install -r src/PcapStatistics/requirements.txt

from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "PcapStatistics").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Execute este notebook dentro da raiz do projeto.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.PcapStatistics import PcapStatisticsDataset, FEATURE_NAMES, discover_pcaps

PCAP_DIR = Path(r"E:\CICDDoS2019\01-12\PCAP")
OUTPUT_DIR = PROJECT_ROOT / "output" / "PcapStatistics" / "01-12"
WINDOW_SECONDS = 1.0  # Pode mudar para 5.0, 10.0 etc.
MAX_PACKETS = None    # None: leitura completa. Exemplo de teste: 100_000.
FLOW_IDLE_TIMEOUT_SECONDS = 600.0
SUBFLOW_GAP_SECONDS = 1.0
PACKET_LENGTH_MODE = "transport_payload"  # Alternativa explícita: "ip".

extractor = PcapStatisticsDataset(
    windowSeconds=WINDOW_SECONDS,
    maxPackets=MAX_PACKETS,
    flowIdleTimeoutSeconds=FLOW_IDLE_TIMEOUT_SECONDS,
    subflowGapSeconds=SUBFLOW_GAP_SECONDS,
    packetLengthMode=PACKET_LENGTH_MODE,
    keepEmptyWindows=True,
    progressEvery=1_000_000,
)


## 2. Input Files

A descoberta identifica PCAP/PCAPNG pelo cabeçalho binário, incluindo arquivos sem extensão como `SAT-01-12-2018_0`.
Todos os arquivos encontrados são intercalados cronologicamente; a ordem dos nomes não define a ordem dos pacotes.
O estado das conexões é preservado entre arquivos. Sobreposições são intercaladas e nenhum pacote é deduplicado.
Cada arquivo deve estar internamente ordenado: regressões de timestamp interrompem a execução.


In [ ]:
pcapFiles = discover_pcaps(PCAP_DIR)
print(f"Arquivos PCAP/PCAPNG: {len(pcapFiles)} | features: {len(FEATURE_NAMES)} | janela: {WINDOW_SECONDS:g}s")


## 3. Global Statistical Dataset

Uma leitura atualiza contadores, médias e dispersões por pacote. O CSV é gravado progressivamente, sem manter o dataset inteiro em memória.
As janelas seguem o epoch UTC: `[início, fim)`. O timestamp do dataset corresponde ao término da janela; são preservadas as datas reais dos PCAPs.
O intervalo exportado vai da primeira à última observação processada. Janelas sem pacotes entre essas bordas são mantidas; não se inventam horas antes/depois da captura.
Janelas vazias não comprovam disponibilidade do sensor: podem representar silêncio ou lacunas de captura.

As 33 features solicitadas são acompanhadas por **6 contagens de valores únicos e 5 entropias** (44 features).
O CSV contém exclusivamente `Timestamp` e as 44 features. A duração da janela está na configuração e no relatório de fórmulas; contadores auxiliares são internos.

**Definições temporais:**

- Forward/backward são determinados pela primeira orientação observada de cada sessão, sem inferir cliente/servidor por porta ou rótulo.
- `Flow_Duration` é a média da duração dos trechos das sessões dentro da janela; IATs usam pares da mesma sessão e janela.
- Contagens, bytes, cabeçalhos e taxas são da janela atual; as taxas usam sua duração nominal.
- As janelas TCP iniciais preservam o primeiro valor observado de cada direção, sem conhecer pacotes futuros.
- `Subflow_*` soma, por sessão na janela, os respectivos totais divididos pela quantidade de subfluxos locais. Com 1s e limiar de gap de 1s, pode coincidir com os totais.
- O modo padrão de comprimento usa payload de transporte; `ip` usa comprimento IPv4 capturado. A escolha consta no relatório.
- Fragmentos permanecem nas contagens IP e entropias, mas não recebem direção TCP/UDP sem remontagem. Comprimentos desconhecidos ficam ausentes.
- Ausências estatísticas são campos vazios; contadores e somas vazios são zero. Não há imputação.

São adaptações documentadas para observação temporal, sem promessa de equivalência numérica com registros finais do CICFlowMeter.


In [ ]:
result = extractor.runFiles(
    pcapFiles=pcapFiles,
    outputPath=OUTPUT_DIR / f"global_windows_{WINDOW_SECONDS:g}s.csv",
    featureReportPath=OUTPUT_DIR / "feature_calculations.csv",
)
print(f"Concluído: {result.packetsProcessed:,} pacotes processados | {result.ipv4Packets:,} IPv4 | {result.windowsWritten:,} janelas")


## 4. Dataset Inspection and Feature Calculations

A inspeção lê somente as primeiras linhas do CSV final. O único relatório gerado é `feature_calculations.csv`, com nome, fórmula, unidade e convenções de cada coluna.
A primeira e a última linha podem representar janelas incompletas. Não são exportadas colunas de borda ou diagnóstico. O resumo da execução permanece em memória, sem relatório adicional.
Para a extração completa mantenha `MAX_PACKETS=None`. Reinicie o kernel após atualizar os módulos e execute o notebook em ordem.


In [ ]:
display(pd.read_csv(result.datasetPath, nrows=5))
featureCalculations = pd.read_csv(result.featureReportPath)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(featureCalculations.loc[featureCalculations["Kind"].eq("feature"), ["Feature", "Formula", "Unit", "Notes"]])
print("Dataset:", result.datasetPath)
print("Cálculos:", result.featureReportPath)


In [1]:
from pathlib import Path
from fractions import Fraction
import re

from scapy.utils import RawPcapReader


Pasta = Path(r"E:\CICDDoS2019\01-12\PCAP")
Padrao = "SAT-01-12-2018_*"


def numero_arquivo(caminho):
    # Aceita nomes sem extensão, .pcap ou .pcapng.
    resultado = re.search(
        r"_(\d+)(?:\.pcap|\.pcapng)?$",
        caminho.name,
        flags=re.IGNORECASE,
    )
    if resultado is None:
        raise ValueError(f"Nome fora do padrão: {caminho.name}")

    return int(resultado.group(1))


def timestamp_ns(leitor, metadata):
    # PCAP: comparação inteira, sem perda de precisão por float.
    if hasattr(metadata, "sec"):
        fator = 1 if leitor.nano else 1000
        return metadata.sec * 1_000_000_000 + metadata.usec * fator

    # PCAPNG: considera a resolução registrada na captura.
    if metadata.tshigh is None or metadata.tslow is None:
        raise ValueError("Foi encontrado um pacote sem timestamp.")

    ticks = (metadata.tshigh << 32) | metadata.tslow
    return Fraction(ticks * 1_000_000_000, metadata.tsresol)


arquivos = sorted(
    (arquivo for arquivo in Pasta.glob(Padrao) if arquivo.is_file()),
    key=numero_arquivo,
)

if not arquivos:
    raise FileNotFoundError(f"Nenhum arquivo encontrado em: {Pasta}")

ultimo_timestamp = None
ultimo_arquivo = None
total_pacotes = 0
total_inversoes = 0

for arquivo in arquivos:
    print(f"\nLendo: {arquivo.name}", flush=True)

    anterior = None
    quantidade = 0
    inversoes = 0
    primeira_inversao = None
    inversao_entre_arquivos = False

    with RawPcapReader(str(arquivo)) as leitor:
        for _, metadata in leitor:
            atual = timestamp_ns(leitor, metadata)
            quantidade += 1

            if quantidade == 1:
                inversao_entre_arquivos = (
                    ultimo_timestamp is not None
                    and atual < ultimo_timestamp
                )

            if anterior is not None and atual < anterior:
                inversoes += 1
                if primeira_inversao is None:
                    primeira_inversao = quantidade

            anterior = atual

    print(f"  Pacotes: {quantidade:,}")
    print(f"  Inversões dentro do arquivo: {inversoes:,}")

    if primeira_inversao is not None:
        print(f"  Primeira inversão no pacote: {primeira_inversao:,}")

    if inversao_entre_arquivos:
        print(
            f"  INVERSÃO ENTRE ARQUIVOS: {arquivo.name} começa "
            f"antes do último pacote de {ultimo_arquivo}"
        )

    if quantidade > 0:
        ultimo_timestamp = anterior
        ultimo_arquivo = arquivo.name

    total_pacotes += quantidade
    total_inversoes += inversoes + int(inversao_entre_arquivos)

print(f"\nTotal de pacotes verificados: {total_pacotes:,}")
print(f"Total de inversões: {total_inversoes:,}")

if total_pacotes == 0:
    print("Nenhum pacote encontrado; não foi possível verificar a ordem.")
elif total_inversoes == 0:
    print("OK: timestamps em ordem cronológica na sequência verificada.")
else:
    print("ATENÇÃO: a sequência possui timestamps fora de ordem.")


Lendo: SAT-01-12-2018_0
  Pacotes: 267,138
  Inversões dentro do arquivo: 0

Lendo: SAT-01-12-2018_01
  Pacotes: 403,278
  Inversões dentro do arquivo: 0

Lendo: SAT-01-12-2018_02
  Pacotes: 403,133
  Inversões dentro do arquivo: 0

Lendo: SAT-01-12-2018_03
  Pacotes: 403,201
  Inversões dentro do arquivo: 0

Lendo: SAT-01-12-2018_04
  Pacotes: 402,860
  Inversões dentro do arquivo: 0

Lendo: SAT-01-12-2018_05
  Pacotes: 403,114
  Inversões dentro do arquivo: 0

Lendo: SAT-01-12-2018_06
  Pacotes: 403,029
  Inversões dentro do arquivo: 0

Lendo: SAT-01-12-2018_07
  Pacotes: 403,352
  Inversões dentro do arquivo: 0

Lendo: SAT-01-12-2018_08
  Pacotes: 403,165
  Inversões dentro do arquivo: 0

Lendo: SAT-01-12-2018_09
  Pacotes: 403,245
  Inversões dentro do arquivo: 0

Lendo: SAT-01-12-2018_010
  Pacotes: 403,455
  Inversões dentro do arquivo: 0

Lendo: SAT-01-12-2018_011
  Pacotes: 403,097
  Inversões dentro do arquivo: 0

Lendo: SAT-01-12-2018_012
  Pacotes: 402,862
  Inversões dentr

KeyboardInterrupt: 